# Quickstart

[**User Guide**](https://millerbrainobservatory.github.io/LBM-Suite2p-Python/user_guide.html) | 
[**Supported Filetypes**](https://millerbrainobservatory.github.io/mbo_utilities/array_types.html#quick-reference) | 
[**API Reference**](https://millerbrainobservatory.github.io/LBM-Suite2p-Python/api.html) | 
[**MBO Hub**](https://millerbrainobservatory.github.io/)

Suite2p-based calcium imaging pipeline for Light Beads Microscopy data.

The only required parameter is the input data.

`input_data` can be any filepath or Lazy Array supported by [imread](https://millerbrainobservatory.github.io/mbo_utilities/user_guide.html#reading-and-writing-data).

In [ ]:
import lbm_suite2p_python as lsp

ops = {
    "diameter": 2,
    "anatomical_only": 4,
    "accept_all_cells": True,
    "spatial_hp_cp": 0,
    "denoise": 0,
    "two_step_registration": 0,
}

input_data = r"\\rbo-s1\S1_DATA\lbm\jdemas\bi_hemisphere"
save_path = r"\\rbo-s1\S1_DATA\lbm\jdemas\bi_hemisphere\results"

results = lsp.pipeline(
    input_data=input_data,      # path to .zarr, .tiff, or .bin file
    save_path=save_path,        # default: save next to input file
    ops=ops,                    # default: use MBO-optimized parameters
    planes=1,                   # process single plane (1-indexed)
    keep_reg=True,              # default: keep data.bin (registered binary)
    keep_raw=False,             # default: delete data_raw.bin after processing
    force_reg=False,            # default: skip if already registered
    force_detect=False,         # default: skip if stat.npy exists
    num_frames=500,             # default: use all frames
    dff_window_size=None,       # default: auto-calculate from tau and framerate
    dff_percentile=20,          # default: 20th percentile for baseline
    dff_smooth_window=None,     # default: auto-calculate from tau and framerate
    reader_kwargs={
        "fix_phase": True,      # default: MboRawArrays correct phase by default
        "use_fft": True,        # default: MboRawArrays use FFT default
    },
)

Loading input data...
  Input: \\rbo-s1\S1_DATA\lbm\jdemas\bi_hemisphere


Counting frames:   0%|          | 0/3 [00:00<?, ?it/s]

  Loaded as: MboRawArray

Dataset info:
  Shape: (1176, 30, 1202, 1305)
  Frames: 1176
  Planes: 30
  Dimensions: 1202 x 1305
  Phase correction: True
  FFT subpixel: True
  Frame rate: 2.18 Hz
  ROIs: 9
Importing suite2p packages...

Processing plan:
  Planes: [7]
  Output: \\rbo-s1\S1_DATA\lbm\jdemas\bi_hemisphere\results

Processing plane 7/30
  Writing binary (1176 frames, 1202x1305)...


Saving data_raw.bin:   0%|          | 0/500 [00:00<?, ?it/s]

  Running Suite2p pipeline...
NOTE: applying default C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
----------- REGISTRATION
Reference frame, 32.58 sec.
Registered 500/500 in 50.87s
----------- Total 89.21 sec
----------- ROI DETECTION
Binning movie in chunks of length 03
Binned movie of size [166,1198,1303] created in 1.66 sec.
>>>> CELLPOSE finding masks in max_proj


model_type argument is not used in v4.0.1+. Ignoring this argument...


!NOTE! diameter set to 2.00 for cell detection with cellpose


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


>>>> 556 masks detected, median diameter = 6.08 
Detected 556 ROIs, 324.44 sec
After removing overlaps, 556 ROIs remain
----------- Total 328.58 sec.
----------- EXTRACTION
Masks created, 1.81 sec.
Extracted fluorescence from 556 ROIs in 500 frames, 6.59 sec.
----------- Total 8.51 sec.
----------- CLASSIFICATION
['skew', 'npix_norm', 'compact']
----------- SPIKE DECONVOLUTION
----------- Total 0.02 sec.
  Applying default diameter filter (4-35 µm, pixel_size=4.58 µm/px)
  Applying cell filters...
filter_by_max_diameter: removed 23 ROIs (min=0.9px, max=7.6px)
Saved filtered iscell to \\rbo-s1\S1_DATA\lbm\jdemas\bi_hemisphere\results\plane07_stitched\iscell.npy
apply_filters: 23 total ROIs removed (366/389 cells remaining)
  Saved 14_filter_max_diameter.png
  Computing dF/F...
  Generating plots...
Plotting results for 366 accepted / 190 rejected ROIs
  Completed plane07_stitched in 585.4s

Pipeline complete!
  Processed: 1 planes
  Time: 585.5s
  Output: \\rbo-s1\S1_DATA\lbm\jdemas\bi_

## Load Results

In [ ]:
results = lsp.load_planar_results(ops_files[0])

F = results["F"]           # (n_rois, n_frames)
Fneu = results["Fneu"]     # (n_rois, n_frames)
stat = results["stat"]     # ROI stats
iscell = results["iscell"] # (n_rois, 2)

print(f"ROIs: {len(stat)}, Accepted: {iscell[:, 0].sum():.0f}")

## Compute ΔF/F

In [ ]:
iscell_mask = iscell[:, 0].astype(bool)
F_cells = F[iscell_mask]

dff = lsp.dff_rolling_percentile(
    F_cells,
    window_size=300,  # 10 × tau × framerate
    percentile=20,
)

## Suite2p GUI

In [ ]:
from suite2p import gui
gui.run(statfile=str(ops_files[0].parent / "stat.npy"))

In [2]:
import lbm_suite2p_python as lsp
import matplotlib.pyplot as plt

# basic usage
fig = lsp.plot_3d_rastermap_clusters("D:/demo/results/suite2p")
plt.show()

# with custom clusters
# fig = lsp.plot_3d_rastermap_clusters("D:/demo/results/suite2p", n_clusters=50)

# save to file
# fig = lsp.plot_3d_rastermap_clusters("D:/demo/results/suite2p", save_path="rastermap_clusters.png")


<Figure size 1400x1000 with 0 Axes>

---

## What's Next

- [User Guide](https://millerbrainobservatory.github.io/LBM-Suite2p-Python/user_guide.html) - Parameters, outputs, troubleshooting
- [Grid Search](https://github.com/MillerBrainObservatory/LBM-Suite2p-Python/blob/master/dev/grid_search.ipynb) - Optimize parameters
- [Postprocessing](https://millerbrainobservatory.github.io/LBM-Suite2p-Python/postprocessing.html) - Filters and quality metrics